# ENGE707 Kaggle — Geometry-adversarial eight-class training

**Goal:** classify Cat/Dog × Eye/Face/Mouth/Nose without depending on the original image's width, height, or aspect ratio. Preserve actual anatomical geometry, hair/texture and RGB colour. This adds (1) randomised aspect-preserving framing, (2) two-view consistency, and (3) a gradient-reversal geometry adversary during training.

**Important:** Adversarial debiasing **discourages**, but cannot guarantee the elimination of, nuisance geometry cues. Some image structure is genuinely predictive; an overly strong penalty can reduce accuracy. Retain the previous `deep_learning.ipynb`, existing checkpoints, and the **0.9490 public Kaggle benchmark**. The ingredients of the winning variable-blend/TTA submission were not supplied; this notebook does not claim to reproduce its weights.

Only the **provided competition training images** are used. No external training images, manual Kaggle test labelling, or hidden label inference. Execute all cells through the preview and tiny smoke test first; start multi-epoch training by setting `RUN_FULL_TRAINING=True` in the final training cell.

In [ ]:
# 1 — Imports, reproducibility and local paths
from pathlib import Path
import json, re, random, sys, platform
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.transforms import InterpolationMode

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
ROOT=Path.cwd()   # open VS Code at your Kaggle repository root
LABELS_CSV=ROOT/'train_labels.csv'  # Confirmed by your VS Code screenshot
SAMPLE_CSV=ROOT/'sample_submission.csv'  # Optional
TRAIN_DIR=ROOT/'train'/'train'  # Confirmed labelled images
TEST_DIR=ROOT/'test'  # Optional; never read for training/preprocessing statistics
AUDIT_DIR=ROOT/'outputs'/'preflight_audit'
OUT=ROOT/'outputs'/'robust_recognition_pipeline'
OUT.mkdir(parents=True,exist_ok=True)
assert LABELS_CSV.is_file(),f'Missing {LABELS_CSV}'
assert TRAIN_DIR.is_dir(),f'Missing {TRAIN_DIR}'
print('Python:',platform.python_version(),'Torch:',torch.__version__,'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Training images:',TRAIN_DIR)

In [ ]:
# 2 — Dataset, class mapping and submission schema (sample labels are placeholders!)
labels=pd.read_csv(LABELS_CSV)
assert {'image_id','class'}.issubset(labels.columns)
assert labels[['image_id','class']].notna().all().all()
assert not labels.image_id.duplicated().any(),'Duplicate IDs in train_labels.csv'
labels['image_id']=labels.image_id.astype(int)
CLASSES=sorted(labels['class'].unique())
assert len(CLASSES)==8, f'Expected 8 classes, got {CLASSES}'
CLASS_TO_IDX={name:i for i,name in enumerate(CLASSES)}
IDX_TO_CLASS={i:name for name,i in CLASS_TO_IDX.items()}
labels['target']=labels['class'].map(CLASS_TO_IDX)
print('Number of training images:',len(labels),'\nClasses:',CLASSES)
print(labels['class'].value_counts().reindex(CLASSES).to_string())
if SAMPLE_CSV.is_file():
    sample_submission=pd.read_csv(SAMPLE_CSV)
    assert {'image_id','class'}.issubset(sample_submission.columns)
    print('Kaggle test rows:',len(sample_submission),'— sample_submission.class is NOT ground truth.')
print('Class mapping:',CLASS_TO_IDX)
(OUT/'class_to_idx.json').write_text(json.dumps(CLASS_TO_IDX,indent=2),encoding='utf-8')

In [ ]:
# 3 — File index: must search the TRAIN folder only (train/test IDs overlap).
IMAGE_EXT={'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
def index_images(root):
    paths={};duplicates=[];other=[]
    for p in root.rglob('*'):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXT:continue
        match=re.search(r'(\d+)$',p.stem)
        if not match:other.append(str(p));continue
        k=int(match.group(1))
        if k in paths:duplicates.append((k,str(paths[k]),str(p)))
        else:paths[k]=p
    if duplicates:raise ValueError(f'Duplicate image IDs in image folder: {duplicates[:3]}')
    if other:raise ValueError(f'Image names have no trailing numeric ID: {other[:3]}')
    return paths
image_index=index_images(TRAIN_DIR)
missing=sorted(set(labels.image_id)-set(image_index))
assert not missing,f'{len(missing)} images missing; first IDs {missing[:10]}'
labels['path']=labels.image_id.map(lambda i:str(image_index[i]))
print('Images matched to labels:',len(labels),'Folder files:',len(image_index))

In [ ]:
# 4 — Data inspection (labels + sampled original image geometry)
counts=labels['class'].value_counts().reindex(CLASSES)
fig,ax=plt.subplots(figsize=(9,3));counts.plot(kind='bar',ax=ax)
ax.set(title='Training examples per class',xlabel='Class',ylabel='Count')
plt.xticks(rotation=40,ha='right');plt.tight_layout();plt.show()

SAMPLE_INSPECTION=min(1500,len(labels))
observed=[]
for r in labels.sample(SAMPLE_INSPECTION,random_state=SEED).itertuples(index=False):
    with Image.open(r.path) as im:
        observed.append((r.image_id,r[1],im.width,im.height,im.mode))
geometry=pd.DataFrame(observed,columns=['image_id','class','width','height','mode'])
print('Sizes (sample):\n',geometry[['width','height']].describe().to_string())
print('Modes:',geometry['mode'].value_counts().to_dict())
print('Most common sizes:',geometry.groupby(['width','height']).size().sort_values(ascending=False).head(8).to_dict())

In [ ]:
# 5 — Preview labelled examples (don't use test image labels)
fig,axes=plt.subplots(2,4,figsize=(12,6))
for ax,cls in zip(axes.ravel(),CLASSES):
    r=labels[labels['class']==cls].sample(n=1,random_state=SEED).iloc[0]
    with Image.open(r.path) as image:
        ax.imshow(image.convert('RGB'))
    ax.set_title(f'{cls} / {r.image_id}',fontsize=10)
    ax.axis('off')
plt.tight_layout();plt.show()

In [ ]:
# 5b — Colour/greyscale/edges: inspect what preprocessing preserves and discards.
from PIL import ImageFilter, ImageEnhance
fig, axes=plt.subplots(4,4,figsize=(11,10))
for row,cls in enumerate([CLASSES[0],CLASSES[1],CLASSES[4],CLASSES[5]]):
    record=labels[labels['class']==cls].sample(1,random_state=SEED).iloc[0]
    with Image.open(record.path) as im:
        rgb=ImageOps.pad(im.convert('RGB'),(224,224),method=Image.Resampling.BICUBIC,color=(124,116,104))
        gray=ImageOps.grayscale(rgb)
        edges=gray.filter(ImageFilter.FIND_EDGES)
        contrast=ImageEnhance.Contrast(gray).enhance(1.8)
    for col,(title,view) in enumerate([('RGB',rgb),('Greyscale',gray),('Edges',edges),('Contrast x1.8',contrast)]):
        axes[row,col].imshow(view,cmap='gray' if col else None)
        axes[row,col].set_title(f'{cls}: {title}',fontsize=9)
        axes[row,col].axis('off')
plt.tight_layout();plt.show()
print('Diagnostic visualizations only. Keep RGB as baseline; greyscale/edge maps are not substitutes for the original images.')

In [ ]:
# 5c — Training-only image statistics by class (sampled, no target leakage).
# These are diagnostics, not model input features or proof of class-specific biology.
probe=labels.groupby('class',group_keys=False).sample(n=min(90,int(labels['class'].value_counts().min())),random_state=SEED)
stats=[]
for r in probe.itertuples(index=False):
    with Image.open(r.path) as im:
        arr=np.asarray(im.convert('RGB').resize((64,64)),dtype=np.float32)/255.0
    rgb_mean=arr.mean(axis=(0,1))
    luminosity=(0.299*arr[:,:,0]+0.587*arr[:,:,1]+0.114*arr[:,:,2])
    gx=np.diff(luminosity,axis=1); gy=np.diff(luminosity,axis=0)
    stats.append((r[1],*rgb_mean,float(luminosity.std()),float(np.mean(gx**2)+np.mean(gy**2))))
pixel_stats=pd.DataFrame(stats,columns=['class','mean_R','mean_G','mean_B','gray_contrast','edge_energy'])
print('Sampled image statistics; substantial class differences may signal unintended background/source artefacts:')
print(pixel_stats.groupby('class').mean().round(4).to_string())
pixel_stats.to_csv(OUT/'sampled_pixel_statistics.csv',index=False)

In [ ]:
# 5d — Look for shortcut opportunities: dimensions and lighting versus labels.
# This probes the original labelled training files, NOT hidden test data.
geometry['aspect']=geometry['width']/geometry['height']
geometry['pixels']=geometry['width']*geometry['height']
print('Class-wise sampled image geometry (possible dataset artefacts):')
print(geometry.groupby('class')[['width','height','aspect','pixels']].median().round(2))
print('Class-wise sampled RGB, grayscale contrast and edge-energy:')
print(pixel_stats.groupby('class')[['mean_R','mean_G','mean_B','gray_contrast','edge_energy']].mean().round(3))
fig,axes=plt.subplots(1,2,figsize=(12,4))
for label,group in geometry.groupby('class'):
    axes[0].hist(group['aspect'].clip(0,4),bins=25,alpha=.35,label=label)
axes[0].set(title='Original image aspect ratios (sampled)',xlabel='width / height')
for label,group in pixel_stats.groupby('class'):
    axes[1].hist(group['gray_contrast'],bins=18,alpha=.35,label=label)
axes[1].set(title='Brightness variation across samples',xlabel='grayscale standard deviation')
axes[1].legend(fontsize=7,ncol=2)
plt.tight_layout();plt.show()
print('Different distributions may reveal shortcut cues; they do not alone demonstrate the model uses those cues.')

In [ ]:
# 6 — Evaluation safeguards only: group already-detected exact copies, WITHOUT manual review.
# Similar images (different crop, brightness or zoom) are kept for learning.
GROUP_FILE=AUDIT_DIR/'group_assignments.csv'
if GROUP_FILE.is_file():
    group_df=pd.read_csv(GROUP_FILE)
    assert {'image_id','class','group_id'}.issubset(group_df.columns)
    assert not group_df.image_id.duplicated().any()
    assert set(group_df.image_id.astype(int))==set(labels.image_id), 'Audit group map covers a different dataset.'
    group_df['image_id']=group_df.image_id.astype(int)
    assert labels.set_index('image_id')['class'].equals(group_df.set_index('image_id')['class'].reindex(labels.image_id))
    labels=labels.merge(group_df[['image_id','group_id']],on='image_id',validate='one_to_one')
    print('Loaded automatic exact-duplicate grouping; manual review is NOT required.')
else:
    labels['group_id']=labels.image_id.astype(str)
    print('Audit group file missing. Splitting by label only; exact copies may cross split.')
print('Image count:',len(labels),'Groups:',labels.group_id.nunique(),
      'Multi-image groups:',int((labels.groupby('group_id').size()>1).sum()))

In [ ]:
# 7 — Once-only group-aware 80/20 train/validation split (held-out validation).
SPLIT_CSV=OUT/'train_val_split.csv'
if SPLIT_CSV.exists():
    split=pd.read_csv(SPLIT_CSV)
    assert {'image_id','split'}.issubset(split.columns)
    assert set(split.image_id)==set(labels.image_id),'Saved split does not match current labels.'
    assert not split.image_id.duplicated().any()
    assert set(split['split'])=={'train','val'}
    labels=labels.merge(split[['image_id','split']],on='image_id',validate='one_to_one')
    print('Reused prior saved split:',SPLIT_CSV)
else:
    if labels.group_id.nunique()<len(labels):
        sgkf=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
        tr_ix,va_ix=next(sgkf.split(labels,labels.target,groups=labels.group_id))
    else:
        tr_ix,va_ix=train_test_split(np.arange(len(labels)),test_size=0.2,random_state=SEED,stratify=labels.target)
    labels['split']='train'
    labels.loc[va_ix,'split']='val'
    labels[['image_id','split']].to_csv(SPLIT_CSV,index=False)
train_df=labels[labels.split=='train'].copy().reset_index(drop=True)
val_df=labels[labels.split=='val'].copy().reset_index(drop=True)
assert set(train_df.group_id).isdisjoint(set(val_df.group_id)), 'Known duplicate crosses splits. Remove the OLD prepared_pipeline/train_val_split.csv and regenerate.'
print('Split sizes:',len(train_df),len(val_df),'Validation proportion:',round(len(val_df)/len(labels),4))
print(pd.crosstab(labels['class'],labels['split']).reindex(CLASSES))
print('Class balance (train vs validation fractions):')
print(pd.DataFrame({'train':train_df['class'].value_counts(normalize=True),'validation':val_df['class'].value_counts(normalize=True)}).reindex(CLASSES).round(3))

## 8 — Model configuration and framing

Start with a **pretrained**, manageable baseline. Set `MODEL_NAME` to your confirmed best *individual* architecture if known. The default `convnext_tiny` is a testable starting backbone, **not** a claim that it produced the 0.9490 blend. Torchvision pretrained weights are fetched on the first run. All images are RGB and normalised with ImageNet statistics matching these pretrained models. Random reflected framing retains the entire image rather than creating fixed, easily recognisable padding bars. Reflective seams can still carry cues, hence separate validation stability tests remain important.

In [ ]:
# 8a — Config: keep the original Kaggle paths and saved split from above.
MODEL_NAME = 'convnext_tiny'        # torchvision: convnext_tiny / resnet50 / efficientnet_v2_s
PRETRAINED = True                 # keep True for the real Kaggle run
INITIAL_WEIGHTS = None           # Optional identical-architecture checkpoint for this NEW two-head model
INPUT_SIZE = 224
BATCH_SIZE = 4                   # 2 views -> 8 tensors/forward; lower to 2 if GPU memory is tight
NUM_WORKERS = 0                  # Windows/Jupyter-safe; tune later
GRAY_PROB = 0.04                # Small perturbation; preserve RGB for the main signal
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)
INTERP = InterpolationMode.BICUBIC

print('Backbone:', MODEL_NAME, '| pretrained:', PRETRAINED)
print('Classes:', CLASSES, '| Kaggle target: 8 classes')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 8b — Label-independent square framing; full original image remains visible.
# Reflect padding removes fixed solid bars; random canvas scale + offset weakens frame shortcuts.
# No aggressive RandomResizedCrop: an already tight 10x10 crop must retain its feature.
class VariableReflectCanvas:
    def __init__(self, training=True):
        self.training = training
    def __call__(self, im):
        a = np.asarray(im.convert('RGB'))
        h, w = a.shape[:2]
        factor = random.uniform(1.0, 1.28) if self.training else 1.0
        side = max(max(w, h), int(round(max(w, h) * factor)))
        pad_x, pad_y = side-w, side-h
        left = random.randint(0, pad_x) if self.training else pad_x//2
        top = random.randint(0, pad_y) if self.training else pad_y//2
        pads = ((top,pad_y-top),(left,pad_x-left),(0,0))
        return Image.fromarray(np.pad(a,pads,mode='reflect' if min(h,w)>1 else 'edge'))

train_transform = T.Compose([
    VariableReflectCanvas(training=True),
    T.Resize((INPUT_SIZE, INPUT_SIZE), interpolation=INTERP, antialias=True),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([T.RandomAffine(degrees=8, translate=(.03,.03), scale=(.9,1.1),
                                  interpolation=INTERP, fill=(124,116,104))],p=0.6),
    T.RandomApply([T.ColorJitter(brightness=.12, contrast=.12,
                                 saturation=.10, hue=.012)],p=0.6),
    T.RandomGrayscale(p=GRAY_PROB),
    T.ToTensor(), T.Normalize(mean=MEAN,std=STD)
])
val_transform = T.Compose([
    VariableReflectCanvas(training=False),
    T.Resize((INPUT_SIZE,INPUT_SIZE),interpolation=INTERP,antialias=True),
    T.ToTensor(), T.Normalize(mean=MEAN,std=STD)
])
print('Training / validation transforms configured (RGB and aspect-preserving).')

## 9 — Adversary targets: original file geometry, training data only

The nuisance targets are the **original** file size and aspect ratio, never class labels. Only the adversary's **training target** receives these values: the main model inputs remain the transformed RGB pixels. Binning boundaries are computed from the training split alone. Metadata is cached to avoid reopening all images on later runs.

In [ ]:
# 9a — Original dimensions; cache for 63,660 images after the first run.
DIMENSIONS_CSV = OUT / 'original_image_dimensions.csv'
if DIMENSIONS_CSV.is_file():
    dimensions = pd.read_csv(DIMENSIONS_CSV)
    assert {'image_id','width','height'}.issubset(dimensions.columns)
    assert set(dimensions.image_id.astype(int)) == set(labels.image_id), 'Dimension cache does not match dataset.'
else:
    dimensions = []
    for j, r in enumerate(labels.itertuples(index=False),1):
        with Image.open(r.path) as im: w,h=im.size
        dimensions.append((int(r.image_id),int(w),int(h)))
        if j % 15000==0: print('Original sizes indexed:',j)
    dimensions = pd.DataFrame(dimensions,columns=['image_id','width','height'])
    dimensions.to_csv(DIMENSIONS_CSV,index=False)

# train_df/val_df are defined in the saved split cell above.
train_df=train_df.drop(columns=['width','height'],errors='ignore').merge(dimensions,on='image_id',validate='one_to_one')
val_df=val_df.drop(columns=['width','height'],errors='ignore').merge(dimensions,on='image_id',validate='one_to_one')
for table in (train_df,val_df):
    table['log_size'] = np.log2(np.maximum(table.width,table.height).astype(float))
    table['log_aspect'] = np.log(table.width / table.height)
assert (train_df[['width','height']] > 0).all().all()
print('Original geometry cached for',len(dimensions),'images; never passed as classifier inputs.')

In [ ]:
# 9b — Binned nuisance targets, boundaries based ONLY on training data.
def quartile_edges(series):
    return np.unique(np.quantile(series.astype(float),[.25,.50,.75]))

size_edges=quartile_edges(train_df['log_size'])
aspect_edges=quartile_edges(train_df['log_aspect'])
if not len(size_edges): raise ValueError('Could not build size bins.')
if not len(aspect_edges): raise ValueError('Could not build aspect bins.')

for table in (train_df,val_df):
    table['size_bin'] = np.searchsorted(size_edges,table.log_size.to_numpy(),side='right').astype(int)
    table['aspect_bin'] = np.searchsorted(aspect_edges,table.log_aspect.to_numpy(),side='right').astype(int)
N_SIZE = len(size_edges)+1
N_ASPECT = len(aspect_edges)+1
print('Size nuisance classes:',N_SIZE,'distribution:',train_df.size_bin.value_counts().sort_index().to_dict())
print('Aspect nuisance classes:',N_ASPECT,'distribution:',train_df.aspect_bin.value_counts().sort_index().to_dict())
print('The network sees no size/aspect metadata directly; only the geometry heads have these targets.')

In [ ]:
# 9c — Two independently augmented views of each training image; one deterministic validation view.
class FacialViews(Dataset):
    def __init__(self,table,transform,two_views):
        self.table=table.reset_index(drop=True)
        self.transform=transform
        self.two_views=two_views
    def __len__(self): return len(self.table)
    def __getitem__(self,i):
        r=self.table.iloc[i]
        with Image.open(r.path) as im:
            im=im.convert('RGB')
            x1=self.transform(im)
            if self.two_views: x2=self.transform(im)
        if self.two_views:
            return x1,x2,int(r.target),int(r.size_bin),int(r.aspect_bin)
        return x1,int(r.target),int(r.size_bin),int(r.aspect_bin)

train_ds=FacialViews(train_df,train_transform,True)
val_ds=FacialViews(val_df,val_transform,False)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
a,b,y,size_bin,aspect_bin=next(iter(train_loader))
print('Two-view batch:',a.shape,b.shape,'target',y.shape,'geometry labels:',size_bin.shape,aspect_bin.shape)
assert a.shape == b.shape and a.shape[1]==3 and torch.isfinite(a).all() and torch.isfinite(b).all()

In [ ]:
# 9d — Visual check: the same image is presented with different scale, position and lighting.
row = train_df.sample(1,random_state=SEED).iloc[0]
with Image.open(row.path) as im: im = im.convert('RGB')
fig, axes=plt.subplots(2,4,figsize=(11,6))
for ax in axes.flat:
    view=train_transform(im)
    img=(view*torch.tensor(STD).view(3,1,1)+torch.tensor(MEAN).view(3,1,1)).clamp(0,1)
    ax.imshow(img.permute(1,2,0).numpy());ax.axis('off')
fig.suptitle(f'Label stays {row["class"]}; nuisance framing changes per view')
plt.tight_layout();plt.show()

## 10 — Backbone + gradient reversal

The main eight-class head learns classification. The **adversary** tries to recover *original* size and aspect-ratio bins from the same pooled features. The custom gradient reversal function changes the **backbone** gradient's sign, but the geometry-head weights still minimise their normal prediction loss. At inference the geometry heads are unused. Using a frozen backbone permanently would make the penalty ineffective; after one optional head-only epoch, the backbone is unfrozen.

In [ ]:
# 10a — Backbone is configurable; use pretrained weights unless running a local smoke test.
# torchvision is already part of this environment; no timm requirement for these choices.
from torchvision import models
import torch.nn.functional as F

class GradientReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx,x,coefficient):
        ctx.coefficient=float(coefficient)
        return x.view_as(x)
    @staticmethod
    def backward(ctx,grad_output):
        return -ctx.coefficient * grad_output, None

def create_backbone(name,pretrained=True):
    options={
        'convnext_tiny':(models.convnext_tiny,models.ConvNeXt_Tiny_Weights.DEFAULT),
        'resnet50':(models.resnet50,models.ResNet50_Weights.DEFAULT),
        'efficientnet_v2_s':(models.efficientnet_v2_s,models.EfficientNet_V2_S_Weights.DEFAULT),
    }
    if name not in options: raise ValueError(f'Unsupported model {name}. Available: {list(options)}')
    factory,weights=options[name]
    net=factory(weights=weights if pretrained else None)
    if name=='resnet50':
        dim=net.fc.in_features
        net.fc=nn.Identity()
    else:
        dim=net.classifier[-1].in_features
        net.classifier[-1]=nn.Identity()  # retains ConvNeXt LayerNorm and Flatten
    return net,dim

class DebiasedFacialNet(nn.Module):
    def __init__(self,backbone_name,pretrained,n_size,n_aspect,n_classes):
        super().__init__()
        self.backbone,dim=create_backbone(backbone_name,pretrained)
        self.classifier=nn.Sequential(nn.Dropout(.15),nn.Linear(dim,n_classes))
        self.size_head=nn.Sequential(nn.Linear(dim,128),nn.ReLU(),nn.Linear(128,n_size))
        self.aspect_head=nn.Sequential(nn.Linear(dim,128),nn.ReLU(),nn.Linear(128,n_aspect))
    def forward(self,x,grl_coefficient=0.0,geometry_heads=False):
        feats=self.backbone(x)
        logits=self.classifier(feats)
        if not geometry_heads: return logits
        rev=GradientReverse.apply(feats,grl_coefficient)
        return logits,self.size_head(rev),self.aspect_head(rev)

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net=DebiasedFacialNet(MODEL_NAME,PRETRAINED,N_SIZE,N_ASPECT,len(CLASSES)).to(DEVICE)
print('Total parameters:',f'{sum(p.numel() for p in net.parameters()):,}')
if INITIAL_WEIGHTS is not None:
    ckpt=torch.load(INITIAL_WEIGHTS,map_location='cpu',weights_only=False)
    # This requires identical class order and architecture, no silent partial loads.
    assert ckpt['classes']==CLASSES and ckpt['model_name']==MODEL_NAME
    net.load_state_dict(ckpt['model'],strict=True)
    print('Restored compatible initial geometry-adversarial checkpoint.')
else:
    print('Backbone initialisation:', 'pretrained weights' if PRETRAINED else 'random weights (smoke test only)')
    print('This does NOT restore the 0.9490 variable-blend models.')

In [ ]:
# 10b — Losses: correct labels for both views + prediction agreement + GRL geometry adversary.
def two_view_losses(logits,size_logits,aspect_logits,target,size_target,aspect_target,consistency_weight):
    n=len(target)
    primary=0.5*(F.cross_entropy(logits[:n],target)+F.cross_entropy(logits[n:],target))
    p1=F.softmax(logits[:n],dim=1);p2=F.softmax(logits[n:],dim=1)
    reference=(p1+p2).detach()/2
    consistency=0.5*(F.kl_div(F.log_softmax(logits[:n],dim=1),reference,reduction='batchmean')+
                     F.kl_div(F.log_softmax(logits[n:],dim=1),reference,reduction='batchmean'))
    geom_size=0.5*(F.cross_entropy(size_logits[:n],size_target)+F.cross_entropy(size_logits[n:],size_target))
    geom_aspect=0.5*(F.cross_entropy(aspect_logits[:n],aspect_target)+F.cross_entropy(aspect_logits[n:],aspect_target))
    nuisance=(geom_size+geom_aspect)/2
    # GRL changes only the backbone gradient; the nuisance heads still minimise this loss.
    total=primary+consistency_weight*consistency+nuisance
    return total,primary.detach(),consistency.detach(),nuisance.detach()

# Sign test: head gradient unchanged; shared-feature gradient inverted and scaled.
head=nn.Linear(2,2,bias=False)
x0=torch.tensor([[1.0,-1.0]],requires_grad=True)
loss0=F.cross_entropy(head(x0),torch.tensor([0]))
normal_grad=torch.autograd.grad(loss0,x0,retain_graph=False)[0]
x1=x0.detach().clone().requires_grad_(True)
loss1=F.cross_entropy(head(GradientReverse.apply(x1,0.2)),torch.tensor([0]))
reversed_grad=torch.autograd.grad(loss1,x1)[0]
assert torch.allclose(reversed_grad,-0.2*normal_grad,atol=1e-7), 'Gradient sign reversal failed'
print('Gradient reversal autograd test passed.')

## 11 — Optimisation settings

On your 8 GB GPU, begin with 224×224, batch 4 and accumulation 4. **Two augmented views** cost more compute than ordinary supervised training. First epoch (optional) trains the new heads only; later epochs train the full backbone with a smaller learning rate. The nuisance reversal coefficient is warmed up, not set to an extreme value. Save both latest and best checkpoints; select based on unchanged validation images.

In [ ]:
# 11a — Training controls. Nothing long-running starts until RUN_FULL_TRAINING=True below.
MAX_EPOCHS=16
HEAD_ONLY_EPOCHS=1
GRAD_ACCUM_STEPS=4
BACKBONE_LR=2e-5
HEAD_LR=2e-4
WEIGHT_DECAY=0.01
CONSISTENCY_WEIGHT=0.25
MAX_GRL_COEFF=0.12        # Moderate adversarial pressure; 1.0+ is NOT recommended initially
GRL_WARMUP_EPOCHS=5
EARLY_STOP_PATIENCE=5
MAX_HOURS=5.0
BEST_FILE=OUT/'best_geometry_adversarial.pt'
LAST_FILE=OUT/'last_geometry_adversarial.pt'
AMP=(DEVICE.type=='cuda')

optimizer=torch.optim.AdamW([
    {'params':net.backbone.parameters(),'lr':BACKBONE_LR},
    {'params':list(net.classifier.parameters())+list(net.size_head.parameters())+
              list(net.aspect_head.parameters()),'lr':HEAD_LR},
],weight_decay=WEIGHT_DECAY)
scaler=torch.amp.GradScaler('cuda',enabled=AMP)
print('Full train prepared; AMP=',AMP,'batch=',BATCH_SIZE,'accum=',GRAD_ACCUM_STEPS)
print('Checkpoint folder:',OUT)

In [ ]:
# 11b — One training epoch, validation, and checkpoint functions.
import os, time
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score

def set_backbone_trainable(trainable):
    for p in net.backbone.parameters(): p.requires_grad_(trainable)

def grl_schedule(epoch):
    if epoch<HEAD_ONLY_EPOCHS: return 0.0
    progress=(epoch-HEAD_ONLY_EPOCHS+1)/max(1,GRL_WARMUP_EPOCHS)
    return MAX_GRL_COEFF*min(1.0,progress)

def train_epoch(epoch):
    net.train()
    set_backbone_trainable(epoch>=HEAD_ONLY_EPOCHS)
    if epoch<HEAD_ONLY_EPOCHS: net.backbone.eval()  # hold BatchNorm running stats fixed in head-only epoch
    coefficient=grl_schedule(epoch)
    optimizer.zero_grad(set_to_none=True)
    total_loss=total_ce=total_con=total_geom=seen=0.0
    for step,(x1,x2,y,sz,ar) in enumerate(train_loader):
        y=y.to(DEVICE,non_blocking=True);sz=sz.to(DEVICE,non_blocking=True);ar=ar.to(DEVICE,non_blocking=True)
        x=torch.cat((x1,x2),dim=0).to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type,enabled=AMP):
            logits,sl,al=net(x,grl_coefficient=coefficient,geometry_heads=True)
            loss,ce,con,geom=two_view_losses(logits,sl,al,y,sz,ar,CONSISTENCY_WEIGHT)
        scaled_loss=loss/GRAD_ACCUM_STEPS
        scaler.scale(scaled_loss).backward()
        if (step+1)%GRAD_ACCUM_STEPS==0 or (step+1)==len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(net.parameters(),1.0)
            scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True)
        n=len(y)
        total_loss+=float(loss.detach())*n;total_ce+=float(ce)*n
        total_con+=float(con)*n;total_geom+=float(geom)*n;seen+=n
    return {'train_loss':total_loss/seen,'train_ce':total_ce/seen,
            'train_consistency':total_con/seen,'train_geometry_loss':total_geom/seen,
            'grl_coefficient':coefficient}

@torch.inference_mode()
def validate():
    net.eval()
    y_true=[];y_pred=[];total=seen=0
    for x,y,sz,ar in val_loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type,enabled=AMP):
            logits=net(x,geometry_heads=False)
            loss=F.cross_entropy(logits,y)
        total+=loss.item()*len(y);seen+=len(y)
        y_true.extend(y.cpu().tolist());y_pred.extend(logits.argmax(1).cpu().tolist())
    return {'val_ce':total/seen,'val_accuracy':accuracy_score(y_true,y_pred),
            'val_balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
            'val_macro_f1':f1_score(y_true,y_pred,average='macro')}

def save_checkpoint(path,epoch,best_accuracy,history):
    payload={'epoch':epoch,'model':net.state_dict(),'optimizer':optimizer.state_dict(),
             'scaler':scaler.state_dict(),'best_accuracy':best_accuracy,
             'history':history,'classes':CLASSES,'model_name':MODEL_NAME,
             'image_size':INPUT_SIZE,'mean':MEAN,'std':STD,
             'size_edges':size_edges,'aspect_edges':aspect_edges,
             'geometry_grl_max':MAX_GRL_COEFF,'consistency_weight':CONSISTENCY_WEIGHT}
    tmp=path.with_suffix('.tmp')
    torch.save(payload,tmp)
    os.replace(tmp,path)

print('Train/validation/checkpoint functions defined. No extended training has run.')

In [ ]:
# 11c — One small TRAINING STEP using the real dataset to check gradients and memory.
# Unlike full training this intentionally does not update optimizer weights.
SMOKE_TEST=True
if SMOKE_TEST:
    net.train()
    bx1,bx2,by,bs,ba=next(iter(train_loader))
    n=min(2,len(by))
    bx=torch.cat((bx1[:n],bx2[:n]),dim=0).to(DEVICE)
    by,bs,ba=by[:n].to(DEVICE),bs[:n].to(DEVICE),ba[:n].to(DEVICE)
    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast(device_type=DEVICE.type,enabled=AMP):
        scores,scores_size,scores_aspect=net(bx,grl_coefficient=0.10,geometry_heads=True)
        loss,ce,cons,geom=two_view_losses(scores,scores_size,scores_aspect,by,bs,ba,CONSISTENCY_WEIGHT)
    loss.backward()
    assert all(torch.isfinite(g.grad).all() for g in net.classifier.parameters() if g.grad is not None)
    print('One-batch smoke test passed. CE',round(float(ce),4),'consistency',round(float(cons),4),
          'geometry',round(float(geom),4))
    optimizer.zero_grad(set_to_none=True)
    del bx,bx1,bx2,scores,scores_size,scores_aspect,loss,by,bs,ba
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
print('Ready to set RUN_FULL_TRAINING=True in the next cell.')

## 12 — Start extended training **only when ready**

This runs locally on your PC when `RUN_FULL_TRAINING=True`. It does not run in ChatGPT or in the background. It performs one optional head-only epoch, progressively activates the adversary, trains the full backbone, validates on the held-out split, and writes checkpoints after each epoch. The **only** model used for inference is the eight-class classifier; auxiliary geometry heads are discarded at inference time.

In [ ]:
# 12 — Set True ONLY when you intend to run the multi-hour experiment.
RUN_FULL_TRAINING=False
RESUME_FROM=None          # Set to LAST_FILE to resume after interruption; leave None for fresh training.

if RUN_FULL_TRAINING:
    history=[];start_epoch=0;best_accuracy=-1.0;best_f1=-1.0;no_improve=0
    if RESUME_FROM is not None:
        saved=torch.load(RESUME_FROM,map_location=DEVICE,weights_only=False)
        assert saved['classes']==CLASSES and saved['model_name']==MODEL_NAME
        net.load_state_dict(saved['model'],strict=True)
        optimizer.load_state_dict(saved['optimizer'])
        scaler.load_state_dict(saved['scaler'])
        history=saved['history'];start_epoch=saved['epoch']+1
        best_accuracy=float(saved['best_accuracy'])
        best_f1=max([r['val_macro_f1'] for r in history if r['val_accuracy']==best_accuracy],default=-1)
        print('Resuming from epoch',start_epoch+1)
    begin=time.monotonic()
    for epoch in range(start_epoch,MAX_EPOCHS):
        train_metrics=train_epoch(epoch)
        val_metrics=validate()
        result={'epoch':epoch+1,**train_metrics,**val_metrics}
        history.append(result)
        improved=(result['val_accuracy']>best_accuracy+1e-6 or
                  (abs(result['val_accuracy']-best_accuracy)<=1e-6 and result['val_macro_f1']>best_f1+1e-6))
        if improved:
            best_accuracy=result['val_accuracy'];best_f1=result['val_macro_f1']
            no_improve=0
            save_checkpoint(BEST_FILE,epoch,best_accuracy,history)
        else: no_improve+=1
        save_checkpoint(LAST_FILE,epoch,best_accuracy,history)
        pd.DataFrame(history).to_csv(OUT/'geometry_adversarial_history.csv',index=False)
        print(f"Epoch {epoch+1:02d} | train CE {result['train_ce']:.4f} | val accuracy {result['val_accuracy']:.4f} "
              f"| macro-F1 {result['val_macro_f1']:.4f} | GRL {result['grl_coefficient']:.3f} | best {best_accuracy:.4f}")
        if no_improve>=EARLY_STOP_PATIENCE:
            print('Early stopping: validation has not improved.');break
        if (time.monotonic()-begin)/3600>=MAX_HOURS:
            print('Time budget reached; latest checkpoint saved.');break
    print('Training completed; best checkpoint:',BEST_FILE)
else:
    print('RUN_FULL_TRAINING=False. No long training started.')

In [ ]:
# 13 — Evaluate the saved best model; auxiliary heads are unused at inference.
# Run after full training. Does NOT access hidden Kaggle test labels.
if BEST_FILE.is_file():
    saved=torch.load(BEST_FILE,map_location=DEVICE,weights_only=False)
    assert saved['classes']==CLASSES and saved['model_name']==MODEL_NAME
    net.load_state_dict(saved['model'],strict=True)
    print('Best checkpoint validation results:',validate())
else:
    print('No saved best checkpoint yet. Train first, then evaluate.')

### Model interpretation and limits

- **Raw size and aspect ratio are never passed as classifier inputs.** They are auxiliary adversarial targets used only while training the shared feature representation.
- **Geometry independence cannot be guaranteed.** The network may infer source dimensions from pixel artefacts, and overly strong debiasing might suppress legitimate shapes. Use the held-out validation score (and later controlled view tests) to select the penalty.
- A low adversary accuracy **alone** does not prove debiasing worked: the adversary could simply be weak. Robustness and label accuracy are essential cross-checks.
- For fair comparison, keep the existing split and record settings. The 0.9490 Kaggle **public** score belongs to a variable-blend TTA submission, not this model.
- No test-image manual labelling or other Kaggle rule-bypassing method is included.